In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
# 2. Preparar los datos de entrenamiento
# Separamos las características (X) de la variable objetivo (y)
# Eliminamos 'ID' porque no sirve para predecir, y 'SeriousDlqin2yrs' de X
X = train_df.drop(['SeriousDlqin2yrs', 'ID'], axis=1)
y = train_df['SeriousDlqin2yrs']

In [4]:
# 3. División para validación interna (Hold-out set)
# Usamos stratify=y para mantener la proporción de clases (importante en fraude/riesgo)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
# 4. Crear el Pipeline de procesamiento y modelado
# - SimpleImputer: Rellena los valores nulos (NaN) con la mediana de la columna
# - StandardScaler: Escala los datos (opcional para árboles, pero buena práctica)
# - GradientBoostingClassifier: El modelo predictivo
pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler', StandardScaler()),
                     ('classifier', GradientBoostingClassifier
                      (n_estimators=100,    # Número de árboles
                       learning_rate=0.1,   # Tasa de aprendizaje
                       max_depth=3,         # Profundidad de cada árbol
                       random_state=42))])

In [6]:
# 5. Entrenar y Validar (con el conjunto de validación)
print("Entrenando con conjunto de validación...")
pipeline.fit(X_train, y_train)

Entrenando con conjunto de validación...


,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [7]:
# Predecimos probabilidades (necesario para AUC) usando [:, 1] para la clase positiva
y_val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

In [8]:
# Predecimos probabilidades (necesario para AUC) usando [:, 1] para la clase positiva
y_val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

In [9]:
# Calculamos el AUC
auc_score = roc_auc_score(y_val, y_val_pred_proba)
print(f"Validation AUC Score: {auc_score:.4f}")

Validation AUC Score: 0.8602


In [10]:
# 6. Entrenamiento Final y Predicción sobre Test
# Ahora entrenamos el modelo con TODOS los datos disponibles (X completo)
print("Re-entrenando con todos los datos y generando predicciones...")
pipeline.fit(X, y)

Re-entrenando con todos los datos y generando predicciones...


,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [11]:
# Preparamos los datos de test (quitamos ID)
X_test = test_df.drop(['ID'], axis=1)

In [12]:
# Predecimos probabilidades para el test
test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

In [13]:
# 7. Crear archivo de envío (Submission)
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'SeriousDlqin2yrs': test_pred_proba})

submission_file = 'submission.csv'
submission.to_csv(submission_file, index=False)

print(f"¡Listo! Archivo generado: {submission_file}")
print(submission.head())

¡Listo! Archivo generado: submission.csv
       ID  SeriousDlqin2yrs
0  129460          0.096409
1  134018          0.014887
2   86523          0.014993
3  138466          0.009412
4  143905          0.008771
